#### 工具内部访问 Context

上下文提供在调用时传入的不可变配置信息。将其用于用户 ID、会话详情或在对话过程中不应更改的应用特定设置。

In [1]:
from dataclasses import dataclass
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from agent_lab.model_hub import LLM_FAST

USER_DATABASE = {
    'user123': {
        'name': 'Alice Johnson',
        'account_type': 'Premium',
        'balance': 5000,
        'email': 'alice@example.com',
    },
    'user456': {
        'name': 'Bob Smith',
        'account_type': 'Standard',
        'balance': 1200,
        'email': 'bob@example.com',
    },
}

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f'Account holder: {user["name"]}\nType: {user["account_type"]}\nBalance: ${user["balance"]}'
    return 'User not found'

model = init_chat_model(**LLM_FAST)
agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt='You are a financial assistant.',
)

In [2]:
result = agent.invoke(
    {'messages': [{'role': 'user', 'content': "What's my current balance?"}]},
    context=UserContext(user_id='user123'),
)
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

What's my current balance?
================================== Ai Message ==================================

Let me check your account information.
Tool Calls:
  get_account_info (call_00_du2y6rpxYsA9ash47ThwHUvs)
 Call ID: call_00_du2y6rpxYsA9ash47ThwHUvs
  Args:
================================= Tool Message =================================
Name: get_account_info

Account holder: Alice Johnson
Type: Premium
Balance: $5000
================================== Ai Message ==================================

Your current balance is **$5,000.00**. Here are your account details:

- **Account Holder:** Alice Johnson
- **Account Type:** Premium
- **Balance:** $5,000.00

Is there anything else I can help you with?
